# 실습 6: 특성 엔지니어링 + 학습 데이터 준비

추출된 숫자형 특성을 스케일링하고, 학습/테스트 데이터를 분할합니다.
10주차 모델 학습에 바로 사용할 수 있는 데이터를 저장합니다.

**실행 방법**: JupyterLab에서 셀을 위에서부터 `Shift+Enter`로 순차 실행

**사전 준비**: `preprocessing.ipynb`를 먼저 실행해 `csic2010_cleaned.csv` 생성

**다음 단계**: `approach_preview.ipynb`

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle
import os

print("=" * 65)
print("  [실습 6] 특성 엔지니어링 + 학습 데이터 준비")
print("=" * 65)

# 전처리된 데이터 로드
# Jupyter / 스크립트 양쪽에서 동작하도록 SCRIPT_DIR을 결정
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

data_path = os.path.join(SCRIPT_DIR, "csic2010_cleaned.csv")
if not os.path.exists(data_path):
    print(f"\n  !! csic2010_cleaned.csv 파일이 없습니다.")
    print(f"  >> 먼저 preprocessing.py를 실행하세요.")
    raise SystemExit('필수 파일이 없어 중단합니다')

df = pd.read_csv(data_path)
print(f"\n  데이터 로드: {df.shape[0]:,}행 x {df.shape[1]}열")

  [실습 6] 특성 엔지니어링 + 학습 데이터 준비

  데이터 로드: 9,999행 x 44열


## Step 1: 라벨 인코딩

In [2]:
print(f"\n{'─' * 65}")
print("  Step 1: 라벨 인코딩")
print(f"{'─' * 65}")

# 이진 분류 라벨 (Normal=0, Anomalous=1)
df["is_attack"] = (df["label"] == "Anomalous").astype(int)

normal_count = (df["is_attack"] == 0).sum()
attack_count = (df["is_attack"] == 1).sum()

print(f"\n  정상 (Normal=0):    {normal_count:>8,}건 ({normal_count/len(df)*100:.1f}%)")
print(f"  공격 (Anomalous=1): {attack_count:>8,}건 ({attack_count/len(df)*100:.1f}%)")


─────────────────────────────────────────────────────────────────
  Step 1: 라벨 인코딩
─────────────────────────────────────────────────────────────────

  정상 (Normal=0):       3,584건 (35.8%)
  공격 (Anomalous=1):    6,415건 (64.2%)


## Step 2: 특성(Feature) 선택

In [3]:
print()
print("  " + "─" * 65)
print("  Step 2: 특성 선택")
print("  " + "─" * 65)

# preprocessing.ipynb에서 추출한 숫자형 특성
feature_cols = [
    # 길이 관련
    "url_length", "body_length", "total_length",
    # URL 구조
    "num_params", "path_depth", "has_query", "query_length", "body_num_params",
    # 공격 키워드
    "has_sql", "sql_keyword_count", "has_xss",
    "has_traversal", "traversal_count",
    "has_cmd_injection", "has_crlf",
    # 특수문자 / 공백
    "num_special_chars", "special_char_ratio",
    "num_dots", "num_slashes", "num_spaces",
    # 통계
    "url_entropy", "digit_ratio", "upper_ratio",
]

# 존재하지 않는 컬럼 제외
available_cols = [c for c in feature_cols if c in df.columns]
missing_cols = [c for c in feature_cols if c not in df.columns]

if missing_cols:
    print()
    print(f"  !! 누락된 특성: {missing_cols}")
    print("  >> preprocessing.ipynb를 다시 실행하세요.")

feature_cols = available_cols

print()
print(f"  사용할 특성: {len(feature_cols)}개")

# ============================================================
# 라벨별 평균 비교 (Normal vs Anomalous)
# ============================================================
normal_mask = df["is_attack"] == 0
attack_mask = df["is_attack"] == 1

print()
print("  라벨별 특성 평균 비교 (Normal vs Anomalous)")
print("  " + "─" * 75)
print(f"  {'#':>2s} {'특성':<22s} | {'Normal':>10s} | {'Anomalous':>10s} | {'차이':>9s} | 신호")
print("  " + "─" * 75)

rows = []
for col in feature_cols:
    n_mean = df.loc[normal_mask, col].mean()
    a_mean = df.loc[attack_mask, col].mean()
    diff = a_mean - n_mean
    # 상대 차이 (정상/공격 평균 중 큰 쪽 대비)
    base = max(abs(n_mean), abs(a_mean), 1e-9)
    rel = abs(diff) / base
    rows.append((col, n_mean, a_mean, diff, rel))

# 상대 차이 큰 순으로 정렬
rows.sort(key=lambda x: x[4], reverse=True)

for i, (col, n_mean, a_mean, diff, rel) in enumerate(rows, 1):
    if rel >= 1.0:
        sig = "★★★"
    elif rel >= 0.5:
        sig = "★★ "
    elif rel >= 0.2:
        sig = "★  "
    else:
        sig = "·  "
    sign = "+" if diff >= 0 else ""
    print(f"  {i:>2d} {col:<22s} | {n_mean:>10.3f} | {a_mean:>10.3f} | {sign}{diff:>8.3f} | {sig}")

print("  " + "─" * 75)
print("  범례: ★★★ 차이 100%↑   ★★ 50%↑   ★ 20%↑   · 거의 차이 없음")
print("  >> 상위 특성일수록 정상/공격을 구분하는 신호가 강함")


  ─────────────────────────────────────────────────────────────────
  Step 2: 특성 선택
  ─────────────────────────────────────────────────────────────────

  사용할 특성: 23개

  라벨별 특성 평균 비교 (Normal vs Anomalous)
  ───────────────────────────────────────────────────────────────────────────
   # 특성                     |     Normal |  Anomalous |        차이 | 신호
  ───────────────────────────────────────────────────────────────────────────
   1 has_xss                |      0.000 |      0.027 | +   0.027 | ★★★
   2 has_cmd_injection      |      0.000 |      0.024 | +   0.024 | ★★★
   3 has_crlf               |      0.000 |      0.037 | +   0.037 | ★★★
   4 num_params             |      7.878 |      6.044 |   -1.833 | ★  
   5 sql_keyword_count      |      0.199 |      0.256 | +   0.057 | ★  
   6 has_query              |      0.995 |      0.812 |   -0.183 | ·  
   7 query_length           |    133.851 |    111.750 |  -22.100 | ·  
   8 upper_ratio            |      0.039 |      0.046 | +   0.007 

## Step 3: 결측치 처리 (숫자형 특성)

In [4]:
print(f"\n{'─' * 65}")
print("  Step 3: 결측치 처리")
print(f"{'─' * 65}")

missing = df[feature_cols].isnull().sum()
missing_total = missing.sum()
print(f"\n  결측치 총 개수: {missing_total}")

if missing_total > 0:
    for col in feature_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"    {col}: 중앙값({median_val:.2f})으로 대체")
    print(f"  처리 후 결측치: {df[feature_cols].isnull().sum().sum()}")
else:
    print("  결측치 없음!")


─────────────────────────────────────────────────────────────────
  Step 3: 결측치 처리
─────────────────────────────────────────────────────────────────

  결측치 총 개수: 0
  결측치 없음!


## Step 4: 학습/테스트 데이터 분할 (80:20)

In [5]:
print(f"\n{'─' * 65}")
print("  Step 4: 학습/테스트 데이터 분할 (80:20)")
print(f"{'─' * 65}")

X = df[feature_cols].values
y = df["is_attack"].values

# stratify: 정상/공격 비율을 학습/테스트에 동일하게 유지
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n  학습 데이터: {X_train.shape[0]:,}건")
print(f"  테스트 데이터: {X_test.shape[0]:,}건")
print(f"  학습 공격 비율: {y_train.mean()*100:.1f}%")
print(f"  테스트 공격 비율: {y_test.mean()*100:.1f}%")
print(f"  >> stratify 덕분에 비율이 동일합니다!")


─────────────────────────────────────────────────────────────────
  Step 4: 학습/테스트 데이터 분할 (80:20)
─────────────────────────────────────────────────────────────────

  학습 데이터: 7,999건
  테스트 데이터: 2,000건
  학습 공격 비율: 64.2%
  테스트 공격 비율: 64.1%
  >> stratify 덕분에 비율이 동일합니다!


## Step 5: 특성 스케일링 (StandardScaler)

In [6]:
print(f"\n{'─' * 65}")
print("  Step 5: 특성 스케일링 (StandardScaler)")
print(f"{'─' * 65}")

scaler = StandardScaler()

# ★ 핵심: 학습 데이터로 fit, 테스트 데이터는 transform만!
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled = scaler.transform(X_test)          # transform만

print(f"\n  스케일링 전 (url_length):")
url_idx = feature_cols.index("url_length")
print(f"    학습 평균: {X_train[:, url_idx].mean():.2f}, "
      f"표준편차: {X_train[:, url_idx].std():.2f}")
print(f"\n  스케일링 후:")
print(f"    학습 평균: {X_train_scaled[:, url_idx].mean():.4f}, "
      f"표준편차: {X_train_scaled[:, url_idx].std():.4f}")

print(f"\n  >> 왜 따로 할까?")
print(f"     테스트 데이터 = '미래의 새로운 HTTP 요청'")
print(f"     미래 데이터의 통계를 미리 알 수 없으므로,")
print(f"     학습 데이터의 통계로만 변환 = 데이터 누수(Data Leakage) 방지!")


─────────────────────────────────────────────────────────────────
  Step 5: 특성 스케일링 (StandardScaler)
─────────────────────────────────────────────────────────────────

  스케일링 전 (url_length):
    학습 평균: 150.87, 표준편차: 101.02

  스케일링 후:
    학습 평균: -0.0000, 표준편차: 1.0000

  >> 왜 따로 할까?
     테스트 데이터 = '미래의 새로운 HTTP 요청'
     미래 데이터의 통계를 미리 알 수 없으므로,
     학습 데이터의 통계로만 변환 = 데이터 누수(Data Leakage) 방지!


In [7]:
X_train_scaled

array([[-0.56297415,  0.        , -0.56297415, ..., -0.25749425,
        -0.3450698 , -0.72221083],
       [-0.83024873,  0.        , -0.83024873, ..., -0.75797098,
        -0.92476361, -0.44838558],
       [-0.57287321,  0.        , -0.57287321, ...,  0.72869818,
         2.03321874,  1.02755196],
       ...,
       [ 1.30794791,  0.        ,  1.30794791, ...,  0.83366912,
         0.39600534,  0.08762343],
       [-0.84014779,  0.        , -0.84014779, ..., -0.35856042,
        -0.13729727,  0.05491402],
       [ 1.39703944,  0.        ,  1.39703944, ...,  0.64085797,
         0.38829497,  0.26516614]], shape=(7999, 23))

In [8]:
X_train

array([[9.40000000e+01, 0.00000000e+00, 9.40000000e+01, ...,
        4.62393955e+00, 7.36842105e-02, 2.10526316e-02],
       [6.70000000e+01, 0.00000000e+00, 6.70000000e+01, ...,
        4.44663406e+00, 2.94117647e-02, 2.94117647e-02],
       [9.30000000e+01, 0.00000000e+00, 9.30000000e+01, ...,
        4.97332108e+00, 2.55319149e-01, 7.44680851e-02],
       ...,
       [2.83000000e+02, 0.00000000e+00, 2.83000000e+02, ...,
        5.01050947e+00, 1.30281690e-01, 4.57746479e-02],
       [6.60000000e+01, 0.00000000e+00, 6.60000000e+01, ...,
        4.58813451e+00, 8.95522388e-02, 4.47761194e-02],
       [2.92000000e+02, 0.00000000e+00, 2.92000000e+02, ...,
        4.94220165e+00, 1.29692833e-01, 5.11945392e-02]], shape=(7999, 23))

## Step 6: 클래스 불균형 확인

In [9]:
print(f"\n{'─' * 65}")
print("  Step 6: 클래스 불균형 확인")
print(f"{'─' * 65}")

normal_train = (y_train == 0).sum()
attack_train = (y_train == 1).sum()
ratio = normal_train / attack_train

print(f"\n  학습 데이터:")
normal_bar = "█" * int(normal_train / len(y_train) * 40)
attack_bar = "█" * int(attack_train / len(y_train) * 40)
print(f"    정상:  {normal_train:>8,}건 {normal_bar}")
print(f"    공격:  {attack_train:>8,}건 {attack_bar}")
print(f"\n  정상:공격 비율 = {ratio:.1f}:1")
print(f"  >> class_weight='balanced' 옵션으로 10주차에 대응합니다")


─────────────────────────────────────────────────────────────────
  Step 6: 클래스 불균형 확인
─────────────────────────────────────────────────────────────────

  학습 데이터:
    정상:     2,867건 ██████████████
    공격:     5,132건 █████████████████████████

  정상:공격 비율 = 0.6:1
  >> class_weight='balanced' 옵션으로 10주차에 대응합니다


## Step 7: 10주차용 데이터 저장 (pickle)

In [10]:
print(f"\n{'─' * 65}")
print("  Step 7: 10주차용 데이터 저장")
print(f"{'─' * 65}")

# LLM 테스트용 샘플 (원본 텍스트 포함 — LLM에 HTTP 요청을 보여줄 때 사용)
text_cols = ["method", "url", "url_decoded", "body", "body_decoded",
             "full_text", "label"]
available_text_cols = [c for c in text_cols if c in df.columns]
llm_sample = df[available_text_cols + feature_cols + ["is_attack"]].sample(
    500, random_state=42
)

processed_data = {
    "X_train": X_train_scaled,
    "X_test": X_test_scaled,
    "X_train_raw": X_train,
    "X_test_raw": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_cols": feature_cols,
    "scaler": scaler,
    "llm_sample": llm_sample,
}

output_path = os.path.join(SCRIPT_DIR, "processed_data.pkl")
with open(output_path, "wb") as f:
    pickle.dump(processed_data, f)

print(f"\n  저장 완료: processed_data.pkl")
print(f"    학습 데이터: {X_train_scaled.shape}")
print(f"    테스트 데이터: {X_test_scaled.shape}")
print(f"    특성 수: {len(feature_cols)}개")
print(f"    LLM 테스트 샘플: {len(llm_sample)}건 (원본 텍스트 포함)")


─────────────────────────────────────────────────────────────────
  Step 7: 10주차용 데이터 저장
─────────────────────────────────────────────────────────────────

  저장 완료: processed_data.pkl
    학습 데이터: (7999, 23)
    테스트 데이터: (2000, 23)
    특성 수: 23개
    LLM 테스트 샘플: 500건 (원본 텍스트 포함)


## 전체 요약

In [11]:
print(f"\n{'=' * 65}")
print("  7주차 전체 작업 요약")
print(f"{'=' * 65}")

print("""
  1교시: 웹 보안 이론 + 공격 유형 체험
    -> attack_examples.py, attack_quiz.py

  2교시: CSIC 2010 데이터 로드 + EDA + 시각화
    -> data_load_explore.ipynb, eda_keyword_analysis.ipynb, eda_visualization.ipynb / eda_visualization.py (Streamlit)

  3교시: 전처리 + 특성 엔지니어링 + 학습 데이터 준비
    -> preprocessing.py, feature_engineering.py (지금 이 파일)

  생성된 데이터 파일:
    - csic2010_requests.csv    (파싱된 HTTP 요청 데이터)
    - csic2010_cleaned.csv     (전처리 + 특성 추출 완료)
    - processed_data.pkl       (10주차 모델 학습용)
""")

print("  >> 8주차에서 processed_data.pkl을 로드하여")
print("     통계/ML/LLM 3가지 방법으로 분류 비교 실험을 합니다!")
print(f"{'=' * 65}")


  7주차 전체 작업 요약

  1교시: 웹 보안 이론 + 공격 유형 체험
    -> attack_examples.py, attack_quiz.py

  2교시: CSIC 2010 데이터 로드 + EDA + 시각화
    -> data_load_explore.ipynb, eda_keyword_analysis.ipynb, eda_visualization.ipynb / eda_visualization.py (Streamlit)

  3교시: 전처리 + 특성 엔지니어링 + 학습 데이터 준비
    -> preprocessing.py, feature_engineering.py (지금 이 파일)

  생성된 데이터 파일:
    - csic2010_requests.csv    (파싱된 HTTP 요청 데이터)
    - csic2010_cleaned.csv     (전처리 + 특성 추출 완료)
    - processed_data.pkl       (10주차 모델 학습용)

  >> 8주차에서 processed_data.pkl을 로드하여
     통계/ML/LLM 3가지 방법으로 분류 비교 실험을 합니다!
